In [ ]:
from lsst.daf.butler import Butler

# A Init Butler

In [ ]:
repo = "main"
butler = Butler(repo)

## B - Query Collections

In [ ]:

print("Collections :")
for collection in butler.collections.query("*"):
    print(collection)

## C - Find raow

In [ ]:
print("\nDéfinition de raw :")
print(butler.get_dataset_type("raw"))

In [ ]:
refs = list(
    butler.query_datasets(
        "raw",
        collections=["LSSTCam/raw/all"],
        instrument="LSSTCam",
        find_first=True,
    )
)

print(f"Nombre de datasets raw : {len(refs)}")

In [ ]:
import pandas as pd

rows = []

for ref in refs:
    data_id = ref.dataId

    rows.append({
        "exposure": data_id["exposure"],
        "detector": data_id["detector"],
        "instrument": data_id["instrument"],
        "dataset_id": str(ref.id),
    })

df = pd.DataFrame(rows)

df.head(20)

In [ ]:
df_exposures = (
    df[["exposure", "instrument"]]
    .drop_duplicates()
    .sort_values("exposure")
    .reset_index(drop=True)
)

display(df_exposures)

In [ ]:
print(f"Nombre d'expositions : {len(df_exposures)}")

In [ ]:
refs = list(
    butler.query_datasets(
        "raw",
        collections=["LSSTCam/raw/all"],
        instrument="LSSTCam",
        find_first=True,
    )
)

# Trier par identifiant d'exposition
refs_sorted = sorted(
    refs,
    key=lambda ref: ref.dataId["exposure"]
)

for ref in refs_sorted:
    print(ref.dataId["exposure"], ref.dataId["detector"])

In [ ]:
refs = list(
    butler.query_datasets(
        "raw",
        collections=["LSSTCam/raw/all"],
        where=(
            "instrument = 'LSSTCam' "
            "AND exposure >= 2026021500000 "
            "AND exposure < 2026070100000"
        ),
        find_first=True,
    )
)

df = pd.DataFrame([
    {
        "exposure": ref.dataId["exposure"],
        "detector": ref.dataId["detector"],
        "instrument": ref.dataId["instrument"],
        "dataset_id": str(ref.id),
    }
    for ref in refs
])

df_exposures = (
    df[["exposure", "instrument"]]
    .drop_duplicates()
    .sort_values("exposure")
    .reset_index(drop=True)
)

display(df_exposures)

In [ ]:
import inspect
print(inspect.signature(butler.query_datasets))

In [ ]:
refs = butler.query_datasets(
    "raw",
    collections=["LSSTCam/raw/all"],
    where=(
        "instrument = 'LSSTCam' "
        "AND exposure >= 2026021500000 "
        "AND exposure < 2026070100000"
    ),
    find_first=True,
    order_by=["exposure", "detector"],
    limit=None,
)

print(f"Nombre de datasets raw : {len(refs)}")

In [ ]:
exposures = sorted({
    ref.dataId["exposure"]
    for ref in refs
})

print("Nombre d'expositions distinctes :", len(exposures))
print("Première exposition :", exposures[0])
print("Dernière exposition :", exposures[-1])

In [ ]:
df_raw = pd.DataFrame([
    {
        "exposure": ref.dataId["exposure"],
        "detector": ref.dataId["detector"],
    }
    for ref in refs
])

df_counts = (
    df_raw.groupby("exposure")["detector"]
    .nunique()
    .rename("n_detectors")
    .reset_index()
)

display(df_counts.describe())
display(df_counts.sort_values("exposure").tail(20))